In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [23]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [10]:
!pip install -q sentence-transformers torch transformers scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.4 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [ ]:
import os
# Force single-GPU visibility BEFORE torch is imported, so nothing downstream
# (including any library-internal logic) can decide to multi-GPU wrap.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Visible GPU count: {torch.cuda.device_count()}")  # should be 1

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OPTIONS = ['A', 'B', 'C', 'D', 'E']
MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L12-v2'
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 8
LR = 2e-5
OUTPUT_DIR = './models/mcq_crossencoder_best'

# ============================================================================
# STEP 1: DATA PREPARATION
# ============================================================================

class MCQPairDataset(Dataset):
    """Expands each MCQ row into 5 (prompt, option, label) pairs and
    tokenizes them eagerly so the DataLoader can use the default collate_fn
    on plain tensors (no custom collate / no sentence-transformers needed)."""

    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.pairs = []
        for _, row in df.iterrows():
            prompt = str(row['prompt']).strip()
            correct_answer = row['answer']
            for option_label in OPTIONS:
                option_text = str(row[option_label]).strip()
                label = 1.0 if option_label == correct_answer else 0.0
                self.pairs.append((prompt, option_text, label))
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prompt, option_text, label = self.pairs[idx]
        enc = self.tokenizer(
            prompt, option_text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['label'] = torch.tensor(label, dtype=torch.float)
        return item


def load_data():
    """Load and split data"""
    print("Loading training data...")
    # train_df = pd.read_csv('train.csv')
    # test_df = pd.read_csv('test.csv')

    print(f"\u2713 Train: {len(train_df)} questions")
    print(f"\u2713 Test: {len(test_df)} questions")

    train_data, val_data = train_test_split(
        train_df,
        test_size=0.2,
        random_state=42,
        stratify=train_df['answer']
    )
    train_data = train_data.reset_index(drop=True)
    val_data = val_data.reset_index(drop=True)

    print(f"\u2713 Train split: {len(train_data)}")
    print(f"\u2713 Val split: {len(val_data)}")

    return train_data, val_data, test_df


# ============================================================================
# STEP 2: MANUAL TRAINING LOOP (no sentence-transformers Trainer involved)
# ============================================================================

def train_one_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze(-1)  # (batch,) raw logit, num_labels=1
        loss = loss_fn(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def score_options(model, tokenizer, prompt, option_texts, max_len=MAX_LEN):
    """Score all options for one question in a single batched forward pass."""
    model.eval()
    enc = tokenizer(
        [prompt] * len(option_texts), option_texts,
        truncation=True, max_length=max_len, padding=True, return_tensors='pt'
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits = model(**enc).logits.squeeze(-1)
    scores = torch.sigmoid(logits).cpu().numpy()  # map raw logit -> [0,1] relevance score
    return scores


# ============================================================================
# STEP 3: VALIDATION & METRICS
# ============================================================================

def calculate_map_at_3(predictions):
    """Calculate Mean Average Precision at 3"""
    score = 0
    for pred in predictions:
        if pred['correct'] in pred['predicted']:
            position = pred['predicted'].index(pred['correct']) + 1
            score += 1 / position
    return score / len(predictions)


def validate_model(model, tokenizer, val_df):
    """Validate model using MAP@3"""
    predictions = []
    for _, row in val_df.iterrows():
        prompt = str(row['prompt']).strip()
        correct_answer = row['answer']
        option_texts = [str(row[opt]).strip() for opt in OPTIONS]

        scores = score_options(model, tokenizer, prompt, option_texts)
        top_3_indices = np.argsort(scores)[::-1][:3]
        top_3_labels = [OPTIONS[i] for i in top_3_indices]

        predictions.append({'predicted': top_3_labels, 'correct': correct_answer})

    return calculate_map_at_3(predictions), predictions


# ============================================================================
# STEP 4: INFERENCE ON TEST SET
# ============================================================================

def generate_test_predictions(model, tokenizer, test_df):
    """Generate predictions for test set"""
    predictions = []
    for idx, row in test_df.iterrows():
        prompt = str(row['prompt']).strip()
        option_texts = [str(row[opt]).strip() for opt in OPTIONS]

        scores = score_options(model, tokenizer, prompt, option_texts)
        top_3_indices = np.argsort(scores)[::-1][:3]
        top_3_labels = [OPTIONS[i] for i in top_3_indices]

        predictions.append(' '.join(top_3_labels))

        if (idx + 1) % 100 == 0:
            print(f"\u2713 Processed {idx + 1} test questions")

    return predictions


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("\n" + "="*70)
    print("MCQ SOLVER - CROSS-ENCODER FINE-TUNING (manual training loop)")
    print("="*70)

    train_data, val_data, test_df = load_data()

    print(f"\nLoading base model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1
    ).to(DEVICE)

    print("\nBuilding training pairs...")
    train_ds = MCQPairDataset(train_data, tokenizer)
    print(f"\u2713 Created {len(train_ds)} training pairs")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = max(10, int(total_steps * 0.1))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    # num_labels=1 + 0/1 targets -> binary cross-entropy on the raw logit
    loss_fn = nn.BCEWithLogitsLoss()

    print("\n" + "="*70)
    print("Starting fine-tuning...")
    print("="*70)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    best_map = -1.0
    for epoch in range(EPOCHS):
        avg_loss = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn)
        val_map, _ = validate_model(model, tokenizer, val_data)
        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={avg_loss:.4f} | val_MAP@3={val_map:.4f}")

        # Save the checkpoint with the best validation MAP@3 (the actual
        # competition metric), not just the lowest loss.
        if val_map > best_map:
            best_map = val_map
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
            print(f"  \u2713 New best model saved (val_MAP@3={val_map:.4f})")

    print(f"\n\u2713 Fine-tuning completed! Best val MAP@3: {best_map:.4f}")

    # Reload the best checkpoint for final validation reporting + test inference
    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(DEVICE)

    final_val_map, sample_preds = validate_model(model, tokenizer, val_data)
    print(f"\n\u2713 Final Validation MAP@3: {final_val_map:.4f} ({final_val_map*100:.2f}%)")
    print("\nSample Predictions:")
    for pred in sample_preds[:5]:
        status = "\u2713" if pred['correct'] in pred['predicted'] else "\u2717"
        print(f"{status} Predicted: {pred['predicted']}, Correct: {pred['correct']}")

    print("\n" + "="*70)
    print("GENERATING TEST PREDICTIONS")
    print("="*70)
    test_predictions = generate_test_predictions(model, tokenizer, test_df)

    submission_df = pd.DataFrame({'id': test_df['id'], 'Prediction': test_predictions})
    submission_df.to_csv('submission.csv', index=False)

    print("\n" + "="*70)
    print("SUBMISSION READY")
    print("="*70)
    print(f"\u2713 File: submission.csv")
    print(f"\u2713 Rows: {len(submission_df)}")
    print("\nFirst 5 predictions:")
    print(submission_df.head())
    print("\nLast 5 predictions:")
    print(submission_df.tail())


if __name__ == "__main__":
    main()

PyTorch Version: 2.10.0+cu128
GPU Available: True
GPU Name: Tesla T4
Visible GPU count: 2

MCQ SOLVER - CROSS-ENCODER FINE-TUNING (manual training loop)
Loading training data...
✓ Train: 2000 questions
✓ Test: 500 questions
✓ Train split: 1600
✓ Val split: 400

Loading base model: cross-encoder/ms-marco-MiniLM-L12-v2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Building training pairs...
✓ Created 8000 training pairs

Starting fine-tuning...
Epoch 1/20 | train_loss=0.7885 | val_MAP@3=0.5267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  ✓ New best model saved (val_MAP@3=0.9650)
Epoch 8/20 | train_loss=0.0923 | val_MAP@3=0.9754


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved (val_MAP@3=0.9754)
Epoch 9/20 | train_loss=0.0664 | val_MAP@3=0.9771


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved (val_MAP@3=0.9771)
